# 04 — Hyperparameter Tuning dengan Optuna
**Dataset:** Fruits 360  
**Tujuan Notebook:** Optimasi hyperparameter model terbaik menggunakan Optuna.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import optuna
import pickle
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTPUT_DIR = 'data/processed'
MODEL_DIR  = 'models'
print('Libraries loaded!')

## 2. Load Data

In [ ]:
X_train = np.load(f'{OUTPUT_DIR}/X_train_pca.npy')
X_test  = np.load(f'{OUTPUT_DIR}/X_test_pca.npy')
y_train = np.load(f'{OUTPUT_DIR}/y_train.npy')
y_test  = np.load(f'{OUTPUT_DIR}/y_test.npy')

from sklearn.preprocessing import StandardScaler
with open(f'{MODEL_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Data siap: {X_train_scaled.shape}')

## 3. Tuning SVM dengan Optuna

In [ ]:
def objective_svm(trial):
    """
    Objective function untuk Optuna — mencari kombinasi hyperparameter SVM terbaik.
    Optuna akan mencoba berbagai nilai C, gamma, dan kernel secara otomatis.
    """
    C      = trial.suggest_float('C', 0.1, 100.0, log=True)
    kernel = trial.suggest_categorical('kernel', ['rbf', 'linear', 'poly'])
    gamma  = trial.suggest_categorical('gamma', ['scale', 'auto'])

    model = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)
    # Cross-validation 3-fold untuk estimasi yang lebih stabil
    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy', n_jobs=-1)
    return score.mean()

print('Memulai Optuna tuning untuk SVM...')
study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=30, show_progress_bar=True)

print(f'\nBest SVM params  : {study_svm.best_params}')
print(f'Best CV Accuracy : {study_svm.best_value:.4f}')

## 4. Tuning Random Forest dengan Optuna

In [ ]:
def objective_rf(trial):
    """
    Objective function untuk Random Forest.
    """
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth    = trial.suggest_int('max_depth', 5, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )
    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy', n_jobs=-1)
    return score.mean()

print('Memulai Optuna tuning untuk Random Forest...')
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=30, show_progress_bar=True)

print(f'\nBest RF params   : {study_rf.best_params}')
print(f'Best CV Accuracy : {study_rf.best_value:.4f}')

## 5. Latih Ulang dengan Hyperparameter Terbaik

In [ ]:
# SVM terbaik
best_svm = SVC(**study_svm.best_params, random_state=42)
best_svm.fit(X_train_scaled, y_train)
svm_test_acc = accuracy_score(y_test, best_svm.predict(X_test_scaled))
print(f'SVM tuned Test Accuracy : {svm_test_acc:.4f}')

# Random Forest terbaik
best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42, n_jobs=-1)
best_rf.fit(X_train_scaled, y_train)
rf_test_acc = accuracy_score(y_test, best_rf.predict(X_test_scaled))
print(f'RF  tuned Test Accuracy : {rf_test_acc:.4f}')

# Pilih yang terbaik
if svm_test_acc >= rf_test_acc:
    best_model = best_svm
    best_model_name = 'SVM (Tuned)'
    best_acc = svm_test_acc
else:
    best_model = best_rf
    best_model_name = 'Random Forest (Tuned)'
    best_acc = rf_test_acc

print(f'\nModel terbaik: {best_model_name} dengan accuracy {best_acc:.4f}')

# Simpan model terbaik
with open(f'{MODEL_DIR}/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('Model terbaik disimpan ke models/best_model.pkl')

## 6. Visualisasi Optuna Optimization History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, study, title in zip(axes, [study_svm, study_rf], ['SVM', 'Random Forest']):
    trials_df = study.trials_dataframe()
    ax.plot(trials_df['number'], trials_df['value'], marker='o', markersize=3, color='steelblue')
    ax.axhline(y=study.best_value, color='red', linestyle='--', label=f'Best: {study.best_value:.4f}')
    ax.set_xlabel('Trial')
    ax.set_ylabel('CV Accuracy')
    ax.set_title(f'Optuna Optimization — {title}')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/optuna_history.png', dpi=100)
plt.show()

## 7. Ringkasan
- Optuna berhasil mengoptimasi hyperparameter SVM dan Random Forest dengan 30 trial masing-masing
- Model terbaik disimpan ke `models/best_model.pkl`
- Lanjut ke notebook `05_evaluation.ipynb` untuk evaluasi lengkap